# Retroactively Log Unfrozen Model to MLflow

This notebook logs the trained unfrozen model (saved to DBFS) to MLflow artifacts.

**Problem:** The `finetune_unfrozen.ipynb` notebook saved the model to DBFS but did NOT log it to MLflow artifacts.

**Solution:** Load the model from DBFS and log it to the existing MLflow run.

**Model location:** `/dbfs/FileStore/allen_brain_data/models/unfrozen`

In [ ]:
# Cell 0 - Configuration

import mlflow
import os

# Path where the model was saved during training
MODEL_PATH = "/dbfs/FileStore/allen_brain_data/models/unfrozen"

# MLflow experiment (same as in training)
EXPERIMENT_NAME = "/Users/noel.nosse@grainger.com/histology-brain-segmentation"

# OPTION 1: Get the run ID from the training notebook output or MLflow UI
# After running the cells below to search for the run, you can uncomment and set this:
# RUN_ID = "your-run-id-here"
RUN_ID = None  # Will search for it if None

print(f"Model path: {MODEL_PATH}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Run ID: {RUN_ID or 'Will search automatically'}")

In [ ]:
# Cell 1 - Verify model exists on DBFS

import os

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}")

# List contents to verify
contents = os.listdir(MODEL_PATH)
print(f"Model directory contents ({len(contents)} files):")
for item in sorted(contents)[:20]:  # Show first 20 files
    size = os.path.getsize(os.path.join(MODEL_PATH, item))
    print(f"  {item:50s} {size:>15,} bytes")

if len(contents) > 20:
    print(f"  ... and {len(contents) - 20} more files")

# Check for required files
required_files = ["config.json", "model.safetensors", "preprocessor_config.json"]
missing = [f for f in required_files if f not in contents]
if missing:
    print(f"\n⚠️  WARNING: Missing expected files: {missing}")
else:
    print(f"\n✅ All required model files present")

In [ ]:
# Cell 2 - Find the MLflow run (if RUN_ID not provided)

if RUN_ID is None:
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment is None:
        raise ValueError(f"Experiment '{EXPERIMENT_NAME}' not found")
    
    # Search for runs with "unfrozen" in the name
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="tags.mlflow.runName LIKE 'unfrozen%'",
        order_by=["start_time DESC"],
        max_results=10
    )
    
    if len(runs) == 0:
        raise ValueError("No runs found with 'unfrozen' in the name. Please set RUN_ID manually.")
    
    print(f"Found {len(runs)} 'unfrozen' runs (most recent first):\n")
    for idx, row in runs.iterrows():
        run_id = row['run_id']
        run_name = row.get('tags.mlflow.runName', 'N/A')
        start_time = row['start_time']
        status = row['status']
        num_labels = row.get('params.num_labels', 'N/A')
        mean_iou = row.get('metrics.final_mean_iou', 'N/A')
        print(f"  {idx+1}. {run_name}")
        print(f"     Run ID: {run_id}")
        print(f"     Status: {status}")
        print(f"     Started: {start_time}")
        print(f"     Classes: {num_labels}")
        print(f"     Final mIoU: {mean_iou}")
        print()
    
    # Use the most recent run
    RUN_ID = runs.iloc[0]['run_id']
    print(f"\n✅ Will use most recent run: {RUN_ID}")
    print(f"   If this is not the correct run, stop here and set RUN_ID manually in Cell 0.")
else:
    print(f"Using specified RUN_ID: {RUN_ID}")

In [ ]:
# Cell 4 - Load the model and processor

from transformers import AutoImageProcessor, UperNetForSemanticSegmentation
import os

print("Loading model from DBFS...")

# Load model
model = UperNetForSemanticSegmentation.from_pretrained(MODEL_PATH)
print(f"✅ Loaded model")

# Check if processor was saved with the model
processor_path = os.path.join(MODEL_PATH, "preprocessor_config.json")
if os.path.exists(processor_path):
    # Load processor from saved model
    processor = AutoImageProcessor.from_pretrained(MODEL_PATH)
    print(f"✅ Loaded image processor from saved model")
else:
    # Processor wasn't saved - load from base DINOv2 model
    print(f"⚠️  No preprocessor_config.json found in model directory")
    print(f"   Loading processor from base facebook/dinov2-large model...")
    
    # Load from HuggingFace (or JFrog mirror if configured)
    HF_ENDPOINT = os.environ.get(
        "HF_ENDPOINT",
        "https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote",
    )
    
    try:
        # Try JFrog first
        from huggingface_hub import hf_hub_download
        config_file = hf_hub_download(
            repo_id="facebook/dinov2-large",
            filename="preprocessor_config.json",
            endpoint=HF_ENDPOINT,
            cache_dir="/tmp/dinov2-processor"
        )
        processor = AutoImageProcessor.from_pretrained(os.path.dirname(config_file))
        print(f"✅ Loaded processor from JFrog mirror")
    except Exception as e:
        # Fallback to HuggingFace directly
        print(f"   JFrog failed, trying HuggingFace directly...")
        processor = AutoImageProcessor.from_pretrained("facebook/dinov2-large")
        print(f"✅ Loaded processor from HuggingFace")
    
    # Save processor to model directory for future use
    processor.save_pretrained(MODEL_PATH)
    print(f"   Saved processor to {MODEL_PATH}")

# Print model info
num_params = sum(p.numel() for p in model.parameters())
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel Summary:")
print(f"  Total parameters: {num_params:,}")
print(f"  Trainable parameters: {num_trainable:,}")
print(f"  Number of labels: {model.config.num_labels}")
print(f"  Model type: {model.config.model_type}")

In [ ]:
# Cell 5 - Log model to MLflow

import mlflow.transformers

print(f"Logging model to MLflow run {RUN_ID}...")
print(f"This may take several minutes for a large model.\n")

# Resume the existing run to add artifacts
with mlflow.start_run(run_id=RUN_ID) as run:
    # Log the model with both the model and processor
    # Don't register it - just log as artifacts
    mlflow.transformers.log_model(
        transformers_model={
            "model": model,
            "image_processor": processor
        },
        artifact_path="model",
        task="image-segmentation",
        metadata={
            "source": "retroactive_log",
            "original_path": MODEL_PATH,
            "num_labels": model.config.num_labels,
            "mapping_type": "full"
        }
    )
    
    print(f"✅ Model logged successfully!")
    print(f"\nArtifact path: model")
    print(f"Task: image-segmentation")
    
    # Also log the local DBFS path as a tag for reference
    mlflow.set_tag("dbfs_model_path", MODEL_PATH)
    print(f"\nTagged with DBFS path: {MODEL_PATH}")

In [ ]:
# Cell 6 - Verify the model was logged

# Refresh run info
run = mlflow.get_run(RUN_ID)

# List artifacts
artifacts = mlflow.artifacts.list_artifacts(run_id=RUN_ID)

print(f"Artifacts in run {RUN_ID}:")
for artifact in artifacts:
    print(f"  - {artifact.path}")
    if artifact.is_dir:
        # List contents of directories
        sub_artifacts = mlflow.artifacts.list_artifacts(run_id=RUN_ID, artifact_path=artifact.path)
        for sub in sub_artifacts[:5]:  # Show first 5 items
            print(f"    - {sub.path}")
        if len(sub_artifacts) > 5:
            print(f"    ... and {len(sub_artifacts) - 5} more")

print(f"\n✅ SUCCESS! Model is now available in MLflow.")
print(f"\nYou can now:")
print(f"  1. View the model in MLflow UI")
print(f"  2. Load it with: mlflow.transformers.load_model('runs:/{RUN_ID}/model')")
print(f"  3. Download from DBFS: {MODEL_PATH}")
print(f"\nTo download to your local machine, use:")
print(f"  databricks fs cp -r dbfs:/FileStore/allen_brain_data/models/unfrozen ./models/dinov2-upernet-unfrozen")

In [ ]:
# Cell 6 - Verify the model was logged

# Refresh run info
run = mlflow.get_run(RUN_ID)

# List artifacts
artifacts = mlflow.artifacts.list_artifacts(run_id=RUN_ID)

print(f"Artifacts in run {RUN_ID}:")
for artifact in artifacts:
    print(f"  - {artifact.path}")
    if artifact.is_dir:
        # List contents of directories
        sub_artifacts = mlflow.artifacts.list_artifacts(run_id=RUN_ID, artifact_path=artifact.path)
        for sub in sub_artifacts[:5]:  # Show first 5 items
            print(f"    - {sub.path}")
        if len(sub_artifacts) > 5:
            print(f"    ... and {len(sub_artifacts) - 5} more")

print(f"\n✅ SUCCESS! Model is now available in MLflow.")
print(f"\nYou can now:")
print(f"  1. View the model in MLflow UI")
print(f"  2. Load it with: mlflow.transformers.load_model('runs:/{RUN_ID}/model')")
print(f"  3. Use the registered model: mlflow.transformers.load_model('models:/dinov2-upernet-unfrozen/latest')")

## Summary

This notebook successfully loaded the trained model from DBFS and logged it to MLflow.

**Next Steps:**
1. Update your training notebooks to include `mlflow.transformers.log_model()` before `mlflow.end_run()`
2. Consider creating a helper function to log models consistently across all experiments
3. Document this as a lesson learned

**Prevention:**
Add this code BEFORE `mlflow.end_run()` in future training notebooks:

```python
# Log model to MLflow artifacts
import mlflow.transformers
mlflow.transformers.log_model(
    transformers_model={
        "model": trainer.model,
        "image_processor": processor
    },
    artifact_path="model",
    registered_model_name="your-model-name",
    task="image-segmentation"
)
```